# Q2 Attention Benchmark — Upgraded Version

This notebook upgrades the original Q2 design with:
- multiple neutral filler families
- optional filler variants per article
- optional target-position control
- truncation and token diagnostics
- hard-failure detection
- partial saves after each condition
- trend statistics and Q2 failure-mode classification


In [ ]:

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy"], check=False)

# ============================================================
# Q2 ATTENTION BENCHMARK — UPGRADED VERSION
# ------------------------------------------------------------
# Research question:
# Does performance degrade systematically as input length
# increases, even when task difficulty is held constant?
#
# Main upgrades relative to the prior notebook:
# 1) multiple neutral filler families (not just one repeated block)
# 2) optional multiple filler variants per article
# 3) optional target-position control (front / center / back)
# 4) truncation controls and token-length diagnostics
# 5) condition-by-condition partial saves and run-status logging
# 6) hard-failure detection for malformed / default / invalid runs
# 7) trend statistics and bootstrap CIs for long-minus-short
# 8) Q2 failure-mode classification:
#       - hard_long_context_failure
#       - systematic_length_degradation
#       - degradation_with_pro_epu_drift
#       - length_induced_criterion_drift
#       - length_robust
#       - mixed
# 9) richer focus-excerpt and confidence diagnostics
# ============================================================

import hashlib
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import binomtest
from tqdm.auto import tqdm

import kaggle_benchmarks as kbench

# ---------------------------
# CONFIG
# ---------------------------
INPUT_CSV = "/kaggle/input/datasets/bigdataanalysis1/epu-800/EPU_benchmark_800_balanced.csv"

SEED = 42
N_ROWS = 600
TARGET_MAX_CHARS = 3000

SHORT_TOTAL_CHARS = TARGET_MAX_CHARS
MEDIUM_TOTAL_CHARS = 7000
LONG_TOTAL_CHARS = 13000

BOOT_N = 5000
FILLER_VARIANTS_PER_ARTICLE = 1    # increase to 2 or 3 for stronger robustness analysis
POSITION_MODES = ["center"]        # can also use ["front", "center", "back"] at higher cost

OUT_DIR = Path("/kaggle/working/epu_attention_q2_upgraded_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAMES = [
    # "google/gemini-2.5-flash",
    # "google/gemini-2.5-pro",
    # "anthropic/claude-sonnet-4-6@default",
    # "deepseek-ai/deepseek-v3.2",
    "qwen/qwen3-235b-a22b-instruct-2507",
]

# ---------------------------
# NEUTRAL FILLER PACKS
# ------------------------------------------------------------
# Keep filler non-salient and non-EPU-like. The point is to stress
# length, not salience. Families are deliberately broad and boring.
# ------------------------------------------------------------
FILLER_PACKS = [
    {
        "filler_pack_id": "archive_1",
        "family": "archive_boilerplate",
        "text": (
            "Archive page marker. Edition notice. Section listing. Internal reference line. "
            "Continuation marker. Layout note. Library copy header. Page sequence note. "
            "Archive copy marker. Filing label. Column continuation note. "
        ),
    },
    {
        "filler_pack_id": "catalog_1",
        "family": "catalog_metadata",
        "text": (
            "Catalog record number. Preservation entry. Repository shelf label. "
            "Microfilm reference. Collection note. Indexing code. Retrieval marker. "
            "Descriptive metadata line. Cataloging revision note. Record maintenance entry. "
        ),
    },
    {
        "filler_pack_id": "civic_notice_1",
        "family": "civic_notices",
        "text": (
            "Community bulletin notice. Facility hours update. Routine service announcement. "
            "Scheduling note. Public counter hours listed below. Office contact line. "
            "Clerical processing note. Public notice continuation marker. "
        ),
    },
    {
        "filler_pack_id": "transit_1",
        "family": "transit_summaries",
        "text": (
            "Route summary line. Platform notice. Timetable update marker. "
            "Service bulletin continuation. Station reference label. Transfer note. "
            "Schedule adjustment listing. Passenger information line. "
        ),
    },
    {
        "filler_pack_id": "encyclopedia_1",
        "family": "encyclopedia_neutral",
        "text": (
            "Reference summary entry. General background sentence. Context note. "
            "Non-controversial factual description. Neutral descriptive line. "
            "Reference continuation marker. General information note. "
        ),
    },
]

# ---------------------------
# HELPERS
# ---------------------------
def stable_hash_int(*parts) -> int:
    s = "||".join("" if p is None else str(p) for p in parts)
    return int(hashlib.sha256(s.encode("utf-8")).hexdigest()[:16], 16)

def maybe_int(x):
    if x is None:
        return None
    if isinstance(x, bool):
        return int(x)
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, float):
        if np.isnan(x):
            return None
        return int(round(x))
    s = str(x).strip().lower()
    if s in {"", "none", "null", "nan"}:
        return None
    try:
        return int(round(float(s)))
    except Exception:
        return None

def binary_match(pred, gold):
    pred = maybe_int(pred)
    gold = maybe_int(gold)
    if pred is None or gold is None:
        return None
    return int(pred == gold)

def approx_tokens_from_chars(n_chars: int) -> int:
    try:
        n_chars = int(n_chars)
    except Exception:
        return 0
    return int(math.ceil(max(0, n_chars) / 4.0))

def smart_truncate(text: str, max_chars: int = TARGET_MAX_CHARS) -> str:
    text = "" if text is None else str(text)
    if len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head - 32
    return text[:head] + "\n\n[...TRUNCATED...]\n\n" + text[-tail:]

def choose_filler_pack(article_key: str, variant_idx: int):
    idx = stable_hash_int(article_key, "filler_variant", variant_idx) % len(FILLER_PACKS)
    return FILLER_PACKS[idx]

def make_filler(block_text: str, n_chars: int) -> str:
    if n_chars <= 0:
        return ""
    block_text = str(block_text).strip()
    reps = (n_chars // max(1, len(block_text))) + 3
    filler = ((block_text + " ") * reps)[:n_chars]
    return filler.strip()

def split_filler(total_filler_chars: int, block_text: str, position_mode: str = "center"):
    total_filler_chars = max(0, int(total_filler_chars))
    if total_filler_chars == 0:
        return "", ""

    position_mode = str(position_mode).lower().strip()
    if position_mode == "front":
        pre_chars, post_chars = total_filler_chars, 0
    elif position_mode == "back":
        pre_chars, post_chars = 0, total_filler_chars
    else:
        pre_chars = total_filler_chars // 2
        post_chars = total_filler_chars - pre_chars

    return make_filler(block_text, pre_chars), make_filler(block_text, post_chars)

def excerpt_in_target(excerpt: str, article_text: str) -> int:
    excerpt = "" if excerpt is None else str(excerpt).strip()
    article_text = "" if article_text is None else str(article_text)
    if len(excerpt) < 12:
        return 0
    return int(excerpt in article_text)

def bootstrap_mean_diff_paired(a: np.ndarray, b: np.ndarray, n_boot: int = BOOT_N, seed: int = SEED):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = ~(np.isnan(a) | np.isnan(b))
    a, b = a[mask], b[mask]
    if len(a) == 0:
        return (np.nan, np.nan, np.nan)
    diffs = b - a
    point = float(np.mean(diffs))
    rng = np.random.default_rng(seed)
    idx = np.arange(len(diffs))
    boots = []
    for _ in range(n_boot):
        samp = rng.choice(idx, size=len(idx), replace=True)
        boots.append(float(np.mean(diffs[samp])))
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return point, float(lo), float(hi)

def monotonic_degradation(short_acc, medium_acc, long_acc) -> int:
    vals = [short_acc, medium_acc, long_acc]
    if any(pd.isna(v) for v in vals):
        return 0
    return int(short_acc >= medium_acc >= long_acc and long_acc < short_acc)

def classify_q2_failure_mode(long_acc, short_acc, delta_overall, delta_pos, delta_neg, fp_pull_pp, long_parse_fail_rate, long_valid_rate):
    # hard failure first
    if (
        (not pd.isna(long_parse_fail_rate) and long_parse_fail_rate >= 0.95)
        or (not pd.isna(long_valid_rate) and long_valid_rate <= 0.05)
        or (
            not pd.isna(long_acc) and not pd.isna(short_acc)
            and long_acc <= 0.01 and short_acc >= 0.30
        )
    ):
        return "hard_long_context_failure"

    if any(pd.isna(x) for x in [delta_overall, delta_pos, delta_neg]):
        return "mixed"

    if delta_overall <= -0.02 and delta_pos >= 0.01 and delta_neg <= -0.02 and (not pd.isna(fp_pull_pp)) and fp_pull_pp > 0:
        return "degradation_with_pro_epu_drift"

    if delta_overall <= -0.02 and delta_pos <= -0.01 and delta_neg <= -0.01:
        return "systematic_length_degradation"

    if abs(delta_overall) < 0.02 and delta_pos >= 0.02 and delta_neg <= -0.02 and (not pd.isna(fp_pull_pp)) and fp_pull_pp > 0:
        return "length_induced_criterion_drift"

    if abs(delta_overall) <= 0.02 and abs(delta_pos) <= 0.02 and abs(delta_neg) <= 0.02:
        return "length_robust"

    return "mixed"

# ---------------------------
# LOAD + CLEAN
# ---------------------------
df = pd.read_csv(INPUT_CSV)

required_cols = [
    "article_key",
    "content",
    "gold_epu",
    "disagreement_flag",
    "newspaper",
    "year",
    "content_len",
]

for c in required_cols:
    if c not in df.columns:
        df[c] = np.nan

mask = (
    df["content"].notna()
    & (df["content"].astype(str).str.len() > 300)
    & df["gold_epu"].notna()
)

clean = df.loc[mask].copy()
clean["article_key"] = clean["article_key"].astype(str)
missing_key_mask = clean["article_key"].isin(["nan", "None", ""])
if missing_key_mask.any():
    clean.loc[missing_key_mask, "article_key"] = [f"generated_key_{i}" for i in range(int(missing_key_mask.sum()))]

clean["gold_epu"] = clean["gold_epu"].astype(int)
clean["disagreement_flag"] = clean["disagreement_flag"].fillna(0).astype(int)
clean["content"] = clean["content"].astype(str)
clean["orig_content_len"] = clean["content"].str.len()
clean["content_len"] = clean["content_len"].fillna(clean["orig_content_len"])
clean = clean.drop_duplicates(subset=["article_key"]).reset_index(drop=True)

print(f"Loaded rows: {len(df):,}")
print(f"Clean usable rows: {len(clean):,}")
print("\nClass balance:")
print(clean["gold_epu"].value_counts().sort_index())
print("\nDisagreement rows available:")
print(clean["disagreement_flag"].value_counts().sort_index())

# ---------------------------
# BALANCED SAMPLE
# ---------------------------
def balanced_sample(data: pd.DataFrame, n: int, seed: int = SEED) -> pd.DataFrame:
    pos = data[data["gold_epu"] == 1].copy()
    neg = data[data["gold_epu"] == 0].copy()

    if len(pos) == 0 or len(neg) == 0:
        return data.sample(min(n, len(data)), random_state=seed).reset_index(drop=True)

    n_pos = min(len(pos), n // 2)
    n_neg = min(len(neg), n - n_pos)

    pos_s = pos.sample(n=n_pos, random_state=seed)
    neg_s = neg.sample(n=n_neg, random_state=seed + 1)

    pilot = (
        pd.concat([pos_s, neg_s], axis=0)
        .sample(frac=1, random_state=seed + 2)
        .reset_index(drop=True)
    )
    return pilot

pilot = balanced_sample(clean, N_ROWS, SEED).copy()
pilot["content_for_eval"] = pilot["content"].astype(str).apply(lambda x: smart_truncate(x, TARGET_MAX_CHARS))
pilot["eval_content_len"] = pilot["content_for_eval"].astype(str).str.len()
pilot["was_truncated"] = (pilot["orig_content_len"] > pilot["eval_content_len"]).astype(int)
pilot["truncation_fraction"] = np.where(
    pilot["orig_content_len"] > 0,
    1.0 - (pilot["eval_content_len"] / pilot["orig_content_len"]),
    0.0,
)
pilot["target_est_tokens"] = pilot["eval_content_len"].apply(approx_tokens_from_chars)
pilot["content_len_bin"] = pd.qcut(
    pilot["orig_content_len"].rank(method="first"),
    q=min(4, len(pilot)),
    labels=False,
    duplicates="drop",
)

preview_cols = [
    c for c in [
        "article_key", "gold_epu", "disagreement_flag", "newspaper",
        "year", "orig_content_len", "eval_content_len", "was_truncated"
    ] if c in pilot.columns
]
print("\nPilot preview:")
display(pilot[preview_cols].head())
print("\nPilot label balance:")
print(pilot["gold_epu"].value_counts().sort_index())

# ---------------------------
# BUILD EVAL ROWS
# ------------------------------------------------------------
# Each article can optionally be expanded across:
# - filler variants (different neutral filler families)
# - target positions (front / center / back)
# Defaults are conservative to stay near your current budget.
# ------------------------------------------------------------
def build_eval_rows(pilot_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in pilot_df.iterrows():
        article_key = row["article_key"]
        target_chars = int(row["eval_content_len"])

        short_filler_total = max(0, SHORT_TOTAL_CHARS - target_chars)
        medium_filler_total = max(0, MEDIUM_TOTAL_CHARS - target_chars)
        long_filler_total = max(0, LONG_TOTAL_CHARS - target_chars)

        for filler_variant_id in range(FILLER_VARIANTS_PER_ARTICLE):
            pack = choose_filler_pack(article_key, filler_variant_id)
            filler_text = pack["text"]

            for position_mode in POSITION_MODES:
                short_pre, short_post = split_filler(short_filler_total, filler_text, position_mode)
                medium_pre, medium_post = split_filler(medium_filler_total, filler_text, position_mode)
                long_pre, long_post = split_filler(long_filler_total, filler_text, position_mode)

                eval_row = row.to_dict()
                eval_row.update({
                    "eval_row_id": f"{article_key}__fv{filler_variant_id}__{position_mode}",
                    "filler_variant_id": filler_variant_id,
                    "filler_pack_id": pack["filler_pack_id"],
                    "filler_family": pack["family"],
                    "target_position": position_mode,
                    "short_pre": short_pre,
                    "short_post": short_post,
                    "medium_pre": medium_pre,
                    "medium_post": medium_post,
                    "long_pre": long_pre,
                    "long_post": long_post,
                    "short_total_chars": target_chars + short_filler_total,
                    "medium_total_chars": target_chars + medium_filler_total,
                    "long_total_chars": target_chars + long_filler_total,
                    "short_total_est_tokens": approx_tokens_from_chars(target_chars + short_filler_total),
                    "medium_total_est_tokens": approx_tokens_from_chars(target_chars + medium_filler_total),
                    "long_total_est_tokens": approx_tokens_from_chars(target_chars + long_filler_total),
                })
                rows.append(eval_row)

    return pd.DataFrame(rows)

eval_df = build_eval_rows(pilot).reset_index(drop=True)

pilot_path = OUT_DIR / "pilot_rows_q2_upgraded.csv"
eval_df.to_csv(pilot_path, index=False)

print("\nEvaluation rows:", len(eval_df))
print("Unique articles:", eval_df["article_key"].nunique())
print("Filler families present:")
print(eval_df["filler_family"].value_counts())
print("Target positions present:")
print(eval_df["target_position"].value_counts())

# ---------------------------
# PROMPT + JSON PARSING
# ---------------------------
def build_q2_prompt(article_text: str, context_pre: str = "", context_post: str = "") -> str:
    pre_block = f"NEUTRAL CONTEXT BEFORE TARGET\n{context_pre}\n\n" if context_pre else ""
    post_block = f"\n\nNEUTRAL CONTEXT AFTER TARGET\n{context_post}" if context_post else ""

    return (
        "You are evaluating only the text between TARGET ARTICLE START and TARGET ARTICLE END.\n\n"
        "Task:\n"
        "Decide whether the TARGET ARTICLE expresses ECONOMIC POLICY UNCERTAINTY (EPU).\n\n"
        "Labeling rule:\n"
        "- EPU = 1 only if the TARGET ARTICLE discusses uncertainty about government policy,\n"
        "  policy decisions, possible policy actions, political outcomes affecting policy,\n"
        "  or the economic effects of policy.\n"
        "- EPU = 0 for general economics, elections, markets, war, uncertainty, or foreign events\n"
        "  unless the uncertainty is specifically tied to economic policy.\n\n"
        "IMPORTANT:\n"
        "- Use only the text inside TARGET ARTICLE START and TARGET ARTICLE END for the decision.\n"
        "- The surrounding context blocks are neutral filler and should be ignored.\n"
        "- focus_excerpt must be copied from the TARGET ARTICLE only.\n"
        "- Return ONLY valid JSON.\n"
        "- No markdown. No explanation. No code fence.\n\n"
        "Return exactly these keys:\n"
        "{\n"
        '  "epu": 0 or 1,\n'
        '  "confidence": number between 0 and 1,\n'
        '  "focus_excerpt": "short quote from target article only"\n'
        "}\n\n"
        f"{pre_block}"
        "TARGET ARTICLE START\n"
        f"{article_text}\n"
        "TARGET ARTICLE END"
        f"{post_block}"
    ).strip()

def default_out():
    return {
        "epu": None,
        "confidence": 0.0,
        "focus_excerpt": "",
        "raw_text": "",
        "parse_ok": 0,
        "parse_error": "empty",
    }

def parse_llm_json(raw_text):
    out = default_out()
    out["raw_text"] = "" if raw_text is None else str(raw_text)[:4000]

    if raw_text is None:
        out["parse_error"] = "none_output"
        return out

    text = str(raw_text).strip()
    text = text.replace("```json", "").replace("```", "").strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)
    else:
        out["parse_error"] = "no_json_object"
        return out

    try:
        obj = json.loads(text)
        if not isinstance(obj, dict):
            out["parse_error"] = "json_not_object"
            return out
    except Exception:
        out["parse_error"] = "json_load_error"
        return out

    if "epu" in obj:
        out["epu"] = maybe_int(obj.get("epu"))
    if "confidence" in obj:
        try:
            out["confidence"] = float(obj.get("confidence"))
        except Exception:
            out["confidence"] = 0.0
    if "focus_excerpt" in obj:
        out["focus_excerpt"] = "" if obj.get("focus_excerpt") is None else str(obj.get("focus_excerpt"))[:300]

    out["confidence"] = max(0.0, min(1.0, float(out["confidence"])))
    out["parse_ok"] = int(out["epu"] in {0, 1})
    out["parse_error"] = "ok" if out["parse_ok"] == 1 else "missing_or_invalid_epu"
    return out

def model_json_call(llm, prompt: str):
    try:
        raw = llm.prompt(prompt)
        return parse_llm_json(raw)
    except Exception as e:
        out = default_out()
        out["raw_text"] = f"ERROR: {str(e)[:3000]}"
        out["parse_error"] = "prompt_exception"
        return out

# ---------------------------
# CONDITION EVALUATION
# ---------------------------
CONDITION_CONFIG = {
    "short": {
        "pre_col": "short_pre",
        "post_col": "short_post",
        "total_chars_col": "short_total_chars",
        "total_est_tokens_col": "short_total_est_tokens",
    },
    "medium": {
        "pre_col": "medium_pre",
        "post_col": "medium_post",
        "total_chars_col": "medium_total_chars",
        "total_est_tokens_col": "medium_total_est_tokens",
    },
    "long": {
        "pre_col": "long_pre",
        "post_col": "long_post",
        "total_chars_col": "long_total_chars",
        "total_est_tokens_col": "long_total_est_tokens",
    },
}

records = []
run_status = []
model_handles = {name: kbench.llms[name] for name in MODEL_NAMES}

def save_progress():
    if len(records) > 0:
        pd.DataFrame(records).to_csv(OUT_DIR / "q2_results_detailed_upgraded_partial.csv", index=False)
    if len(run_status) > 0:
        pd.DataFrame(run_status).to_csv(OUT_DIR / "run_status_q2_upgraded.csv", index=False)

for model_name in MODEL_NAMES:
    llm = model_handles[model_name]
    print(f"\nRunning {model_name} ...")

    for condition, cfg in CONDITION_CONFIG.items():
        print(f"  Condition: {condition}")
        condition_start_n = len(records)

        for row in tqdm(eval_df.to_dict(orient="records"), total=len(eval_df), desc=f"{model_name} | {condition}"):
            pre_text = row[cfg["pre_col"]]
            post_text = row[cfg["post_col"]]
            prompt = build_q2_prompt(row["content_for_eval"], pre_text, post_text)
            out = model_json_call(llm, prompt)

            pred = maybe_int(out["epu"])
            gold = maybe_int(row["gold_epu"])

            match = binary_match(pred, gold)
            result = 0.0 if match is None else float(match)
            valid_prediction = int(pred in {0, 1})
            parse_fail = int(out["parse_ok"] == 0)
            excerpt_ok = excerpt_in_target(out["focus_excerpt"], row["content_for_eval"])
            fp = int(gold == 0 and pred == 1) if pred is not None else 0
            fn = int(gold == 1 and pred == 0) if pred is not None else 0

            rec = {
                "llm_name": model_name,
                "condition": condition,
                "eval_row_id": row["eval_row_id"],
                "article_key": row["article_key"],
                "filler_variant_id": row["filler_variant_id"],
                "filler_pack_id": row["filler_pack_id"],
                "filler_family": row["filler_family"],
                "target_position": row["target_position"],
                "gold_epu": row["gold_epu"],
                "disagreement_flag": row["disagreement_flag"],
                "newspaper": row.get("newspaper", ""),
                "year": row.get("year", np.nan),
                "orig_content_len": row["orig_content_len"],
                "eval_content_len": row["eval_content_len"],
                "was_truncated": row["was_truncated"],
                "truncation_fraction": row["truncation_fraction"],
                "target_est_tokens": row["target_est_tokens"],
                "total_chars": row[cfg["total_chars_col"]],
                "total_est_tokens": row[cfg["total_est_tokens_col"]],
                "pred_epu": pred,
                "result": result,
                "valid_prediction": valid_prediction,
                "parse_ok": out["parse_ok"],
                "parse_fail": parse_fail,
                "parse_error": out["parse_error"],
                "confidence": out["confidence"],
                "focus_excerpt": out["focus_excerpt"],
                "excerpt_in_target": excerpt_ok,
                "fp": fp,
                "fn": fn,
                "raw_text": out["raw_text"],
                "prompt_char_len": len(prompt),
                "prompt_est_tokens": approx_tokens_from_chars(len(prompt)),
            }
            records.append(rec)

        # save condition-level outputs immediately
        condition_rows = pd.DataFrame(records[condition_start_n:])
        condition_rows.to_csv(OUT_DIR / f"q2_results_{model_name.replace('/', '__').replace('@', '__')}_{condition}.csv", index=False)

        status_row = {
            "llm_name": model_name,
            "condition": condition,
            "n_rows_expected": len(eval_df),
            "n_rows_completed": len(condition_rows),
            "mean_result": condition_rows["result"].mean() if len(condition_rows) else np.nan,
            "valid_prediction_rate": condition_rows["valid_prediction"].mean() if len(condition_rows) else np.nan,
            "parse_fail_rate": condition_rows["parse_fail"].mean() if len(condition_rows) else np.nan,
        }
        run_status.append(status_row)
        save_progress()

detailed_df = pd.DataFrame(records)
detailed_path = OUT_DIR / "q2_results_detailed_upgraded.csv"
detailed_df.to_csv(detailed_path, index=False)

run_status_df = pd.DataFrame(run_status)
run_status_path = OUT_DIR / "run_status_q2_upgraded.csv"
run_status_df.to_csv(run_status_path, index=False)

print("\nDetailed results shape:", detailed_df.shape)
display(detailed_df.head())

# ---------------------------
# SUMMARIES
# ---------------------------
def summarize_condition_group(g: pd.DataFrame, group_name: str, group_value):
    out = {
        "group_name": group_name,
        "group_value": group_value,
        "n_rows": len(g),
        "accuracy": g["result"].mean() if len(g) else np.nan,
        "accuracy_sd": g["result"].std(ddof=1) if len(g) > 1 else np.nan,
        "valid_prediction_rate": g["valid_prediction"].mean() if len(g) else np.nan,
        "parse_fail_rate": g["parse_fail"].mean() if len(g) else np.nan,
        "mean_confidence": g["confidence"].mean() if len(g) else np.nan,
        "excerpt_in_target_rate": g["excerpt_in_target"].mean() if len(g) else np.nan,
        "fp_rate_among_gold0": g.loc[g["gold_epu"] == 0, "fp"].mean() if (g["gold_epu"] == 0).any() else np.nan,
        "fn_rate_among_gold1": g.loc[g["gold_epu"] == 1, "fn"].mean() if (g["gold_epu"] == 1).any() else np.nan,
        "mean_total_chars": g["total_chars"].mean() if len(g) else np.nan,
        "mean_prompt_chars": g["prompt_char_len"].mean() if len(g) else np.nan,
        "mean_prompt_tokens": g["prompt_est_tokens"].mean() if len(g) else np.nan,
    }
    return out

summary_overall = pd.DataFrame(
    [
        dict({"llm_name": model_name, "condition": condition}, **summarize_condition_group(g, "overall", "all"))
        for (model_name, condition), g in detailed_df.groupby(["llm_name", "condition"], sort=False)
    ]
)
summary_overall_path = OUT_DIR / "q2_summary_overall_upgraded.csv"
summary_overall.to_csv(summary_overall_path, index=False)

summary_by_gold_epu = pd.DataFrame(
    [
        dict({"llm_name": model_name, "condition": condition}, **summarize_condition_group(g, "gold_epu", int(label)))
        for (model_name, condition, label), g in detailed_df.groupby(["llm_name", "condition", "gold_epu"], sort=False)
    ]
)
summary_by_gold_epu_path = OUT_DIR / "q2_summary_by_gold_epu_upgraded.csv"
summary_by_gold_epu.to_csv(summary_by_gold_epu_path, index=False)

summary_by_disagreement = pd.DataFrame(
    [
        dict({"llm_name": model_name, "condition": condition}, **summarize_condition_group(g, "disagreement_flag", int(flag)))
        for (model_name, condition, flag), g in detailed_df.groupby(["llm_name", "condition", "disagreement_flag"], sort=False)
    ]
)
summary_by_disagreement_path = OUT_DIR / "q2_summary_by_disagreement_upgraded.csv"
summary_by_disagreement.to_csv(summary_by_disagreement_path, index=False)

summary_by_filler_family = pd.DataFrame(
    [
        dict({"llm_name": model_name, "condition": condition}, **summarize_condition_group(g, "filler_family", family))
        for (model_name, condition, family), g in detailed_df.groupby(["llm_name", "condition", "filler_family"], sort=False)
    ]
)
summary_by_filler_family_path = OUT_DIR / "q2_summary_by_filler_family_upgraded.csv"
summary_by_filler_family.to_csv(summary_by_filler_family_path, index=False)

summary_by_target_position = pd.DataFrame(
    [
        dict({"llm_name": model_name, "condition": condition}, **summarize_condition_group(g, "target_position", position))
        for (model_name, condition, position), g in detailed_df.groupby(["llm_name", "condition", "target_position"], sort=False)
    ]
)
summary_by_target_position_path = OUT_DIR / "q2_summary_by_target_position_upgraded.csv"
summary_by_target_position.to_csv(summary_by_target_position_path, index=False)

summary_by_truncation = pd.DataFrame(
    [
        dict({"llm_name": model_name, "condition": condition}, **summarize_condition_group(g, "was_truncated", int(flag)))
        for (model_name, condition, flag), g in detailed_df.groupby(["llm_name", "condition", "was_truncated"], sort=False)
    ]
)
summary_by_truncation_path = OUT_DIR / "q2_summary_by_truncation_upgraded.csv"
summary_by_truncation.to_csv(summary_by_truncation_path, index=False)

# ---------------------------
# DEGRADATION TABLE + TREND STATS
# ---------------------------
acc_pivot = summary_overall.pivot(index="llm_name", columns="condition", values="accuracy").reset_index()
valid_pivot = summary_overall.pivot(index="llm_name", columns="condition", values="valid_prediction_rate").reset_index()
parse_pivot = summary_overall.pivot(index="llm_name", columns="condition", values="parse_fail_rate").reset_index()

for col in ["short", "medium", "long"]:
    if col not in acc_pivot.columns:
        acc_pivot[col] = np.nan
    if col not in valid_pivot.columns:
        valid_pivot[col] = np.nan
    if col not in parse_pivot.columns:
        parse_pivot[col] = np.nan

degradation_table = acc_pivot.copy()
degradation_table["medium_minus_short"] = degradation_table["medium"] - degradation_table["short"]
degradation_table["long_minus_short"] = degradation_table["long"] - degradation_table["short"]
degradation_table["long_minus_medium"] = degradation_table["long"] - degradation_table["medium"]
degradation_table["monotonic_degradation_flag"] = degradation_table.apply(
    lambda r: monotonic_degradation(r["short"], r["medium"], r["long"]), axis=1
)

degradation_table = degradation_table.merge(
    valid_pivot.rename(columns={"short": "short_valid_rate", "medium": "medium_valid_rate", "long": "long_valid_rate"}),
    on="llm_name",
    how="left",
)
degradation_table = degradation_table.merge(
    parse_pivot.rename(columns={"short": "short_parse_fail_rate", "medium": "medium_parse_fail_rate", "long": "long_parse_fail_rate"}),
    on="llm_name",
    how="left",
)

# class-conditional pivots
label_pivot = (
    summary_by_gold_epu
    .pivot_table(index="llm_name", columns=["group_value", "condition"], values="accuracy")
)
for lab in [0, 1]:
    for cond in ["short", "medium", "long"]:
        if (lab, cond) not in label_pivot.columns:
            label_pivot[(lab, cond)] = np.nan
label_pivot = label_pivot.reset_index()
label_pivot.columns = ["llm_name"] + [f"gold{lab}_{cond}" for (lab, cond) in label_pivot.columns.tolist()[1:]]

degradation_table = degradation_table.merge(label_pivot, on="llm_name", how="left")
degradation_table["delta_pos_long_minus_short"] = degradation_table["gold1_long"] - degradation_table["gold1_short"]
degradation_table["delta_neg_long_minus_short"] = degradation_table["gold0_long"] - degradation_table["gold0_short"]
degradation_table["fp_pull_pp"] = (1.0 - degradation_table["gold0_long"]) - (1.0 - degradation_table["gold0_short"])
degradation_table["fn_pull_pp"] = (1.0 - degradation_table["gold1_long"]) - (1.0 - degradation_table["gold1_short"])

# paired bootstrap by eval row
trend_rows = []
for model_name, g in detailed_df.groupby("llm_name", sort=False):
    pivot = g.pivot_table(index="eval_row_id", columns="condition", values="result", aggfunc="mean")
    for cond in ["short", "medium", "long"]:
        if cond not in pivot.columns:
            pivot[cond] = np.nan
    slope = np.nan
    if pivot[["short", "medium", "long"]].notna().all(axis=1).sum() > 0:
        y = pivot[["short", "medium", "long"]].mean().to_numpy(dtype=float)
        x = np.array([1.0, 2.0, 3.0], dtype=float)
        slope = float(np.polyfit(x, y, 1)[0])

    d_point, d_lo, d_hi = bootstrap_mean_diff_paired(
        pivot["short"].to_numpy(dtype=float),
        pivot["long"].to_numpy(dtype=float),
        n_boot=BOOT_N,
        seed=SEED,
    )
    trend_rows.append({
        "llm_name": model_name,
        "trend_slope_per_step": slope,
        "long_minus_short_boot_mean": d_point,
        "long_minus_short_boot_ci_lo": d_lo,
        "long_minus_short_boot_ci_hi": d_hi,
    })

trend_stats = pd.DataFrame(trend_rows)
trend_stats_path = OUT_DIR / "q2_trend_stats_upgraded.csv"
trend_stats.to_csv(trend_stats_path, index=False)

degradation_table = degradation_table.merge(trend_stats, on="llm_name", how="left")

degradation_table["failure_mode"] = degradation_table.apply(
    lambda r: classify_q2_failure_mode(
        long_acc=r["long"],
        short_acc=r["short"],
        delta_overall=r["long_minus_short"],
        delta_pos=r["delta_pos_long_minus_short"],
        delta_neg=r["delta_neg_long_minus_short"],
        fp_pull_pp=r["fp_pull_pp"],
        long_parse_fail_rate=r["long_parse_fail_rate"],
        long_valid_rate=r["long_valid_rate"],
    ),
    axis=1,
)

degradation_table_path = OUT_DIR / "q2_degradation_table_upgraded.csv"
degradation_table.to_csv(degradation_table_path, index=False)

# ---------------------------
# ARTICLE × MODEL MATRIX + HARD CASES
# ---------------------------
article_model_matrix = detailed_df.pivot_table(
    index=["eval_row_id", "article_key", "gold_epu", "disagreement_flag", "filler_family", "target_position"],
    columns=["llm_name", "condition"],
    values="result",
    aggfunc="mean"
).reset_index()
article_model_matrix.columns = [
    "__".join([str(c) for c in tup if str(c) != ""]) if isinstance(tup, tuple) else str(tup)
    for tup in article_model_matrix.columns
]
article_model_matrix_path = OUT_DIR / "q2_article_model_matrix_upgraded.csv"
article_model_matrix.to_csv(article_model_matrix_path, index=False)

hardest_rows = (
    detailed_df.groupby(
        ["eval_row_id", "article_key", "gold_epu", "disagreement_flag", "condition", "filler_family", "target_position"],
        sort=False,
    )["result"]
    .agg(["mean", "std", "count", "min", "max"])
    .reset_index()
    .sort_values(["condition", "mean", "std"], ascending=[True, True, False])
)
snippet_df = eval_df[["eval_row_id", "article_key", "content_for_eval"]].copy()
snippet_df["snippet"] = snippet_df["content_for_eval"].astype(str).str.slice(0, 220)
snippet_df = snippet_df[["eval_row_id", "article_key", "snippet"]].drop_duplicates()
hardest_rows = hardest_rows.merge(snippet_df, on=["eval_row_id", "article_key"], how="left")
hardest_rows_path = OUT_DIR / "q2_hardest_rows_upgraded.csv"
hardest_rows.to_csv(hardest_rows_path, index=False)

# ---------------------------
# MANIFEST
# ---------------------------
manifest = pd.DataFrame(
    {
        "file": [
            str(pilot_path),
            str(detailed_path),
            str(run_status_path),
            str(summary_overall_path),
            str(degradation_table_path),
            str(summary_by_gold_epu_path),
            str(summary_by_disagreement_path),
            str(summary_by_filler_family_path),
            str(summary_by_target_position_path),
            str(summary_by_truncation_path),
            str(trend_stats_path),
            str(hardest_rows_path),
            str(article_model_matrix_path),
        ]
    }
)
manifest_path = OUT_DIR / "manifest_q2_upgraded.csv"
manifest.to_csv(manifest_path, index=False)

print("\nSaved files:")
display(manifest)

print("\nOverall summary:")
display(summary_overall.sort_values(["llm_name", "condition"]))

print("\nDegradation table:")
display(degradation_table.sort_values(["long_minus_short", "medium_minus_short"], ascending=[True, True]))
